# Cooling Recommender (Random Forest Only)

This notebook trains **one proper model**: Random Forest.

It predicts `bestTechnique` from:
- `tempC`
- `rh`
- `itLoadKW`
- `electricityPrice`
- `waterPrice`
- `carbonFactor`

It is suitable for beginners and works in **Colab** or **VS Code notebooks**.

In [ ]:
# 1) Install dependencies (run once in Colab)
%pip -q install pandas numpy scikit-learn seaborn matplotlib joblib

In [ ]:
# 2) Imports
import io
import os
import json
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
print('Imports loaded.')

In [ ]:
# 3) Configuration
RANDOM_SEED = 42
TEST_SIZE = 0.20
TARGET_COL = 'bestTechnique'

FEATURE_COLS = [
    'tempC', 'rh', 'itLoadKW',
    'electricityPrice', 'waterPrice', 'carbonFactor'
]

LABEL_FILE_CANDIDATES = ['dataset - Copy.csv', 'dataset_copy.csv', 'dataset.csv']
ARTIFACT_MODEL_PATH = 'cooling_recommender_rf.pkl'
ARTIFACT_REPORT_PATH = 'cooling_recommender_rf_metrics.json'

print('Target column:', TARGET_COL)
print('Feature columns:', FEATURE_COLS)

In [ ]:
# 4) Load dataset (Colab upload first, local fallback)
def _find_file(candidates, names):
    lower_map = {n.lower(): n for n in names}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    for n in names:
        for c in candidates:
            if c.lower() in n.lower():
                return n
    return None

try:
    from google.colab import files
    print('Colab detected: upload your dataset file now...')
    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())
    selected = _find_file(LABEL_FILE_CANDIDATES, uploaded_names)
    if selected is None:
        raise FileNotFoundError('Could not find dataset file in uploads.')
    df = pd.read_csv(io.BytesIO(uploaded[selected]))
    print('Loaded uploaded file:', selected)
except Exception:
    local_names = os.listdir('.')
    selected = _find_file(LABEL_FILE_CANDIDATES, local_names)
    if selected is None:
        raise FileNotFoundError('Dataset file not found locally. Place dataset - Copy.csv in current folder.')
    df = pd.read_csv(selected)
    print('Loaded local file:', selected)

print('Shape:', df.shape)
display(df.head())

In [ ]:
# 5) Validate and clean data
required_cols = FEATURE_COLS + [TARGET_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
assert len(missing_cols) == 0, f'Missing columns: {missing_cols}'

work = df[required_cols].copy()
work = work.drop_duplicates()

for c in FEATURE_COLS:
    work[c] = pd.to_numeric(work[c], errors='coerce')

work[TARGET_COL] = work[TARGET_COL].astype(str).str.strip()
work = work.dropna(subset=required_cols)

print('Cleaned shape:', work.shape)
print('\nMissing values after cleaning:')
print(work.isna().sum())

print('\nClass distribution:')
print(work[TARGET_COL].value_counts())

plt.figure(figsize=(6,4))
sns.countplot(data=work, x=TARGET_COL, order=work[TARGET_COL].value_counts().index)
plt.title('Class Distribution (bestTechnique)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print('\nLeakage note: training uses only the 6 input columns, not any air_/evap_/chill_ outcome columns.')

In [ ]:
# 6) Build train/test split and preprocessing
X = work[FEATURE_COLS].copy()
y = work[TARGET_COL].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y
)

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, FEATURE_COLS)
], remainder='drop')

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

In [ ]:
# 7) Train one proper model: Random Forest
rf_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=500,
        class_weight='balanced',
        min_samples_split=4,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores = cross_val_score(rf_pipeline, X_train, y_train, scoring='f1_macro', cv=cv, n_jobs=-1)
print(f'Baseline RF CV Macro-F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

param_dist = {
    'clf__n_estimators': [300, 500, 700, 900],
    'clf__max_depth': [None, 10, 16, 24, 32],
    'clf__min_samples_split': [2, 4, 6, 10],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features': ['sqrt', 'log2', None]
}

search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1
)

search.fit(X_train, y_train)
final_model = search.best_estimator_

print('\nBest CV Macro-F1:', round(search.best_score_, 4))
print('Best params:', search.best_params_)

In [ ]:
# 8) Evaluate performance on hold-out test set
pred = final_model.predict(X_test)

test_accuracy = accuracy_score(y_test, pred)
test_macro_f1 = f1_score(y_test, pred, average='macro')

print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test Macro-F1: {test_macro_f1:.4f}')
print('\nClassification Report:\n')
print(classification_report(y_test, pred, digits=4))

labels = sorted(y.unique())
cm = confusion_matrix(y_test, pred, labels=labels)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

rf = final_model.named_steps['clf']
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print('\nFeature Importances:')
display(importances)

plt.figure(figsize=(6, 4))
sns.barplot(x=importances.values, y=importances.index)
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# 9) Save artifacts (model + metrics)
artifact = {
    'model': final_model,
    'feature_cols': FEATURE_COLS,
    'target_col': TARGET_COL,
    'random_seed': RANDOM_SEED,
}

joblib.dump(artifact, ARTIFACT_MODEL_PATH)

metrics = {
    'test_accuracy': float(test_accuracy),
    'test_macro_f1': float(test_macro_f1),
    'rows_used': int(len(work)),
    'class_distribution': work[TARGET_COL].value_counts().to_dict(),
    'best_params': search.best_params_,
}

with open(ARTIFACT_REPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved model:', ARTIFACT_MODEL_PATH)
print('Saved metrics:', ARTIFACT_REPORT_PATH)

try:
    from google.colab import files
    files.download(ARTIFACT_MODEL_PATH)
    files.download(ARTIFACT_REPORT_PATH)
except Exception:
    print('Colab download skipped (local mode).')

In [ ]:
# 10) Inference helper (use this in your app)
def recommend_best_technique(tempC, rh, itLoadKW, electricityPrice, waterPrice, carbonFactor, model_artifact=artifact):
    model = model_artifact['model']
    cols = model_artifact['feature_cols']

    sample = pd.DataFrame([{
        'tempC': tempC,
        'rh': rh,
        'itLoadKW': itLoadKW,
        'electricityPrice': electricityPrice,
        'waterPrice': waterPrice,
        'carbonFactor': carbonFactor,
    }])[cols]

    pred = model.predict(sample)[0]
    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(sample)[0]
        labels = model.classes_
        return pred, dict(zip(labels, probs))
    return pred, None

label, probs = recommend_best_technique(32, 55, 1200, 0.14, 1.2, 0.45)
print('Recommended technique:', label)
print('Class probabilities:', probs)

## 11) Robustness Check (Run More Cases)

This section runs many repeated train/test splits so you can prove the model is stable (not lucky on one split).

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import recall_score

# Use best params found earlier; fallback to defaults if not available
if 'search' in globals() and hasattr(search, 'best_params_'):
    tuned_params = {
        'n_estimators': search.best_params_.get('clf__n_estimators', 500),
        'max_depth': search.best_params_.get('clf__max_depth', 10),
        'min_samples_split': search.best_params_.get('clf__min_samples_split', 2),
        'min_samples_leaf': search.best_params_.get('clf__min_samples_leaf', 1),
        'max_features': search.best_params_.get('clf__max_features', 'sqrt')
    }
else:
    tuned_params = {
        'n_estimators': 500,
        'max_depth': 10,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 'sqrt'
    }

print('Robustness model params:', tuned_params)

sss = StratifiedShuffleSplit(n_splits=30, test_size=0.20, random_state=42)

acc_list = []
macro_f1_list = []
class_recall_rows = []

X_all = work[FEATURE_COLS].copy()
y_all = work[TARGET_COL].copy()
classes_sorted = sorted(y_all.unique())

for split_id, (train_idx, test_idx) in enumerate(sss.split(X_all, y_all), start=1):
    X_tr = X_all.iloc[train_idx]
    y_tr = y_all.iloc[train_idx]
    X_te = X_all.iloc[test_idx]
    y_te = y_all.iloc[test_idx]

    model = Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(
            n_estimators=tuned_params['n_estimators'],
            max_depth=tuned_params['max_depth'],
            min_samples_split=tuned_params['min_samples_split'],
            min_samples_leaf=tuned_params['min_samples_leaf'],
            max_features=tuned_params['max_features'],
            class_weight='balanced',
            random_state=42 + split_id,
            n_jobs=-1
        ))
    ])

    model.fit(X_tr, y_tr)
    y_hat = model.predict(X_te)

    acc = accuracy_score(y_te, y_hat)
    mf1 = f1_score(y_te, y_hat, average='macro')
    rec = recall_score(y_te, y_hat, average=None, labels=classes_sorted)

    acc_list.append(acc)
    macro_f1_list.append(mf1)
    class_recall_rows.append(dict(zip(classes_sorted, rec)))

acc_arr = np.array(acc_list)
mf1_arr = np.array(macro_f1_list)

# 95% CI using normal approximation
acc_ci = 1.96 * acc_arr.std(ddof=1) / np.sqrt(len(acc_arr))
mf1_ci = 1.96 * mf1_arr.std(ddof=1) / np.sqrt(len(mf1_arr))

print('\n=== Robustness over 30 repeated splits ===')
print(f'Accuracy mean ± std: {acc_arr.mean():.4f} ± {acc_arr.std(ddof=1):.4f}')
print(f'Accuracy 95% CI: [{acc_arr.mean()-acc_ci:.4f}, {acc_arr.mean()+acc_ci:.4f}]')
print(f'Macro-F1 mean ± std: {mf1_arr.mean():.4f} ± {mf1_arr.std(ddof=1):.4f}')
print(f'Macro-F1 95% CI: [{mf1_arr.mean()-mf1_ci:.4f}, {mf1_arr.mean()+mf1_ci:.4f}]')
print(f'Macro-F1 min/max: {mf1_arr.min():.4f} / {mf1_arr.max():.4f}')

rec_df = pd.DataFrame(class_recall_rows)
print('\nPer-class recall (mean across 30 splits):')
display(rec_df.mean().sort_values(ascending=False).to_frame('mean_recall'))

plt.figure(figsize=(8,4))
plt.plot(mf1_arr, marker='o', linewidth=1)
plt.title('Macro-F1 across 30 repeated splits')
plt.xlabel('Split index')
plt.ylabel('Macro-F1')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12) Hybrid Recommendation + Justification + Future Impact

This section combines your trained model prediction with simulator results for all three techniques, then returns:
- recommended technique
- full comparison table
- justification text
- 1/3/5-year impact projection

In [ ]:
import numpy as np
import pandas as pd

WEIGHTS = {
    'cost': 0.50,
    'emissions': 0.30,
    'water': 0.20
}

def _to_float(value, default=0.0):
    try:
        val = float(value)
        if np.isfinite(val):
            return val
    except Exception:
        pass
    return default

def _normalize(values):
    arr = np.array(values, dtype=float)
    lo = arr.min()
    hi = arr.max()
    if hi - lo < 1e-12:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

def _annualize(hourly_value):
    return _to_float(hourly_value) * 24 * 365

def _prepare_tech_rows(tech_rows):
    rows = []
    for row in tech_rows:
        rows.append({
            'tech': str(row['tech']),
            'feasible': bool(row.get('feasible', True)),
            'energy_kwh': _to_float(row.get('energy_kwh', 0.0)),
            'water_liters': _to_float(row.get('water_liters', 0.0)),
            'cost': _to_float(row.get('cost', 0.0)),
            'emissions_kg': _to_float(row.get('emissions_kg', 0.0)),
            'violations': int(_to_float(row.get('violations', 0)))
        })

    costs = [r['cost'] for r in rows]
    emissions = [r['emissions_kg'] for r in rows]
    waters = [r['water_liters'] for r in rows]

    cost_n = _normalize(costs)
    emissions_n = _normalize(emissions)
    water_n = _normalize(waters)

    for i, r in enumerate(rows):
        r['score'] = (
            WEIGHTS['cost'] * cost_n[i]
            + WEIGHTS['emissions'] * emissions_n[i]
            + WEIGHTS['water'] * water_n[i]
        )
        r['annual_cost'] = _annualize(r['cost'])
        r['annual_emissions_kg'] = _annualize(r['emissions_kg'])
        r['annual_water_liters'] = _annualize(r['water_liters'])

    return rows

def _choose_best(rows):
    feasible_rows = [r for r in rows if r['feasible']]
    pool = feasible_rows if feasible_rows else rows
    return min(pool, key=lambda x: x['score'])

def _build_justification(best, all_rows, model_prediction, model_confidence):
    alternatives = sorted([r for r in all_rows if r['tech'] != best['tech']], key=lambda x: x['score'])
    runner_up = alternatives[0] if alternatives else None

    reasons = [
        f"Model predicted {model_prediction} with confidence {model_confidence:.3f}.",
        f"{best['tech']} has the lowest weighted decision score ({best['score']:.4f}) based on cost/emissions/water.",
        f"Feasibility status for {best['tech']}: {best['feasible']} with violations={best['violations']}."
    ]

    if runner_up is not None:
        reasons.append(
            f"Compared with {runner_up['tech']}, projected annual impact is: cost delta={runner_up['annual_cost'] - best['annual_cost']:.2f}, emissions delta={runner_up['annual_emissions_kg'] - best['annual_emissions_kg']:.2f} kgCO2, water delta={runner_up['annual_water_liters'] - best['annual_water_liters']:.2f} L."
        )

    return reasons

def recommend_with_justification(artifact_obj, scenario, technique_rows):
    model = artifact_obj['model']
    feature_cols = artifact_obj['feature_cols']

    x = pd.DataFrame([scenario])[feature_cols]
    model_pred = model.predict(x)[0]

    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(x)[0]
        labels = model.classes_
        prob_map = {str(k): float(v) for k, v in zip(labels, probs)}
        confidence = float(max(probs))
    else:
        prob_map = None
        confidence = 0.0

    rows = _prepare_tech_rows(technique_rows)
    best = _choose_best(rows)

    justification = _build_justification(best, rows, model_pred, confidence)

    future_impact = {
        'year_1': {
            'cost': best['annual_cost'],
            'emissions_kg': best['annual_emissions_kg'],
            'water_liters': best['annual_water_liters']
        },
        'year_3': {
            'cost': 3 * best['annual_cost'],
            'emissions_kg': 3 * best['annual_emissions_kg'],
            'water_liters': 3 * best['annual_water_liters']
        },
        'year_5': {
            'cost': 5 * best['annual_cost'],
            'emissions_kg': 5 * best['annual_emissions_kg'],
            'water_liters': 5 * best['annual_water_liters']
        }
    }

    response = {
        'model_recommendation': str(model_pred),
        'model_confidence': confidence,
        'model_probabilities': prob_map,
        'final_recommended_technique': best['tech'],
        'comparison_table': rows,
        'justification': justification,
        'future_impact_projection': future_impact
    }
    return response

In [ ]:
scenario_input = {
    'tempC': 32.0,
    'rh': 55.0,
    'itLoadKW': 1200.0,
    'electricityPrice': 0.14,
    'waterPrice': 1.20,
    'carbonFactor': 0.45
}

# Replace these with values from your actual simulator responses for the same scenario.
technique_results = [
    {
        'tech': 'AirEconomizer',
        'feasible': True,
        'energy_kwh': 1300,
        'water_liters': 50,
        'cost': 190,
        'emissions_kg': 580,
        'violations': 0
    },
    {
        'tech': 'Evaporative',
        'feasible': True,
        'energy_kwh': 1150,
        'water_liters': 220,
        'cost': 170,
        'emissions_kg': 520,
        'violations': 0
    },
    {
        'tech': 'ChilledWater',
        'feasible': True,
        'energy_kwh': 1400,
        'water_liters': 160,
        'cost': 205,
        'emissions_kg': 640,
        'violations': 0
    }
]

hybrid_output = recommend_with_justification(artifact, scenario_input, technique_results)

print('Final Recommended Technique:', hybrid_output['final_recommended_technique'])
print('Model Recommendation:', hybrid_output['model_recommendation'])
print('Model Confidence:', round(hybrid_output['model_confidence'], 4))
print('\nJustification:')
for idx, line in enumerate(hybrid_output['justification'], start=1):
    print(f'{idx}. {line}')

print('\nFuture Impact (Year 1):', hybrid_output['future_impact_projection']['year_1'])
print('\nComparison Table:')
display(pd.DataFrame(hybrid_output['comparison_table']).sort_values('score'))